# 🎯 Data Mining — Classification: Core Concepts and Techniques
**Haydar Kılıç | Faculty of Engineering, Artificial Intelligence Engineering | Spring Semester**

---

This notebook is the hands-on Python companion to the lecture slides — theoretical concepts are reproduced exactly, then generalised.

## 📋 Table of Contents
1. [What is Classification?](#1)
2. [Vertebrate Dataset Example](#2)
3. [Confusion Matrix and Performance Metrics](#3)
4. [Decision Tree Classification](#4)
5. [Hunt's Algorithm Simulation](#5)
6. [Attribute Types and Splitting Conditions](#6)
7. [Impurity Measures: Entropy, Gini, Classification Error](#7)
8. [Information Gain and Gain Ratio](#8)
9. [Overfitting and Underfitting](#9)
10. [Model Selection and Cross-Validation](#10)
11. [Model Comparison — Statistical Test](#11)

## ⚙️ Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import confusion_matrix, accuracy_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
print("✅ All libraries loaded successfully!")

---
<a id='1'></a>
## 1. What is Classification?

**Definition:** Every record is represented as an `(x, y)` pair:
- **x** → feature set (independent variable, input)
- **y** → class label (dependent variable, output)

**Task:** Learn a model that predicts a pre-defined label `y` for any given `x`.

### Real-World Examples

In [ ]:
tasks = pd.DataFrame({
    'Task': ['Spam Filtering', 'Tumour Identification', 'Galaxy Classification', 'Credit Risk'],
    'Feature Set (x)': [
        'Email header / body features',
        'MRI scan image features',
        'Telescope image features',
        'Income, marital status, home ownership'
    ],
    'Class Label (y)': [
        'Spam / Not Spam',
        'Malignant / Benign',
        'Elliptical / Spiral / Irregular',
        'Default / No Default'
    ]
})

print("📊 Examples of Classification Tasks")
print("=" * 80)
print(tasks.to_string(index=False))

### Induction and Deduction

- **Induction:** Learning a model from training data → *"Model Learning"*
- **Deduction:** Applying the learned model to new instances → *"Model Application"*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
np.random.seed(42)
X_demo = np.random.randn(50, 2)
y_demo = (X_demo[:, 0] + X_demo[:, 1] > 0).astype(int)
colors = ['#e74c3c' if y == 1 else '#3498db' for y in y_demo]
ax1.scatter(X_demo[:, 0], X_demo[:, 1], c=colors, s=80, alpha=0.8,
            edgecolors='white', linewidth=0.5)
ax1.set_title('📚 Training Set (Induction)\nLearning a model from (x, y) pairs',
              fontsize=12, fontweight='bold')
ax1.set_xlabel('x₁ (Feature 1)')
ax1.set_ylabel('x₂ (Feature 2)')
red_patch  = mpatches.Patch(color='#e74c3c', label='Class 1')
blue_patch = mpatches.Patch(color='#3498db', label='Class 0')
ax1.legend(handles=[red_patch, blue_patch])
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
X_test_demo = np.random.randn(20, 2)
clf_demo = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_demo.fit(X_demo, y_demo)
y_pred_demo = clf_demo.predict(X_test_demo)

xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
Z = clf_demo.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax2.contourf(xx, yy, Z, alpha=0.2, cmap=ListedColormap(['#3498db', '#e74c3c']))
ax2.contour(xx, yy, Z, colors='black', linewidths=1.5, linestyles='--')

test_colors = ['#e74c3c' if y == 1 else '#3498db' for y in y_pred_demo]
ax2.scatter(X_test_demo[:, 0], X_test_demo[:, 1], c=test_colors, s=100,
            alpha=0.9, edgecolors='black', linewidth=1.5, marker='*', zorder=5)
ax2.set_title('🔍 Test Set (Deduction)\nPredicting with the learned model',
              fontsize=12, fontweight='bold')
ax2.set_xlabel('x₁ (Feature 1)')
ax2.set_ylabel('x₂ (Feature 2)')
ax2.legend(handles=[red_patch, blue_patch])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Classification Pipeline: Induction → Deduction',
             fontsize=13, fontweight='bold', y=1.02)
plt.show()

---
<a id='2'></a>
## 2. Vertebrate Dataset Example

We will build and explore the vertebrate dataset from the lecture slides in Python.

In [ ]:
vertebrate_data = {
    'Name': ['human', 'python', 'salmon', 'whale', 'frog',
              'komodo dragon', 'bat', 'pigeon', 'cat'],
    'Body_Temp': ['warm', 'cold', 'cold', 'warm', 'cold',
                  'cold', 'warm', 'warm', 'warm'],
    'Skin_Cover': ['hair', 'scales', 'scales', 'hair', 'none',
                   'scales', 'hair', 'feathers', 'fur'],
    'Live_Birth':  [True,  False, False, True,  False, False, True,  False, True],
    'Aquatic':     [False, False, True,  True,  True,  False, False, False, False],
    'Aerial':      [False, False, False, False, False, False, True,  True,  False],
    'Has_Legs':    [True,  False, False, False, True,  True,  True,  True,  True],
    'Hibernates':  [False, True,  False, False, True,  False, True,  False, False],
    'Class':       ['mammal', 'reptile', 'fish', 'mammal', 'amphibian',
                    'reptile', 'mammal', 'bird', 'mammal']
}

df_vertebrate = pd.DataFrame(vertebrate_data)
print('🦎 Vertebrate Dataset')
print('=' * 80)
print(df_vertebrate.to_string(index=False))
print(f'\n📊 Dataset size: {df_vertebrate.shape[0]} instances, {df_vertebrate.shape[1]} columns')
print(f"🏷️  Classes: {df_vertebrate['Class'].unique()}")

In [ ]:
df_vertebrate['Class_Binary'] = df_vertebrate['Class'].apply(
    lambda x: 'Mammal' if x == 'mammal' else 'Non-Mammal'
)

print('🔵 Binary Classification Conversion (Mammal / Non-Mammal)')
print('=' * 60)
print(df_vertebrate[['Name', 'Class', 'Class_Binary']].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

class_counts = df_vertebrate['Class'].value_counts()
colors_pie = plt.cm.Set3(np.linspace(0, 1, len(class_counts)))
axes[0].pie(class_counts.values, labels=class_counts.index,
            colors=colors_pie, autopct='%1.0f%%', startangle=90)
axes[0].set_title('Multi-Class Distribution', fontweight='bold')

binary_counts = df_vertebrate['Class_Binary'].value_counts()
axes[1].bar(binary_counts.index, binary_counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.5)
axes[1].set_title('Binary Classification Distribution', fontweight='bold')
axes[1].set_ylabel('Count')
for i, (label, val) in enumerate(zip(binary_counts.index, binary_counts.values)):
    axes[1].text(i, val + 0.05, str(val), ha='center', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
print('🦎 Test Instance: Gila Monster')
print('=' * 50)
print('Body Temperature : cold')
print('Skin Cover       : scales')
print('Live Birth       : No')
print('Aquatic          : No')
print('Aerial           : No')
print('Has Legs         : Yes')
print('Hibernates       : Yes')

def decision_tree_predict(body_temp, live_birth):
    """Simulates the simple decision tree from the lecture slides."""
    if body_temp == 'warm':
        return 'Mammal' if live_birth else 'Non-Mammal'
    return 'Non-Mammal'

prediction = decision_tree_predict('cold', False)
print(f'\n✅ Decision Tree Prediction → {prediction}')
print('   Path: Body Temperature=cold → Leaf: Non-Mammal')

---
<a id='3'></a>
## 3. Confusion Matrix and Performance Metrics

$$\\text{Accuracy} = \\frac{f_{11} + f_{00}}{f_{11} + f_{10} + f_{01} + f_{00}}$$

$$\\text{Error Rate} = \\frac{f_{10} + f_{01}}{f_{11} + f_{10} + f_{01} + f_{00}}$$

In [ ]:
le = LabelEncoder()
X_vt = pd.get_dummies(df_vertebrate[['Body_Temp', 'Skin_Cover',
                                       'Live_Birth', 'Aquatic', 'Aerial',
                                       'Has_Legs', 'Hibernates']])
y_vt = le.fit_transform(df_vertebrate['Class_Binary'])

clf_vt = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_vt.fit(X_vt, y_vt)
y_pred_vt = clf_vt.predict(X_vt)

cm = confusion_matrix(y_vt, y_pred_vt)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix', fontweight='bold', fontsize=12)

f11, f10, f01, f00 = cm[1,1], cm[1,0], cm[0,1], cm[0,0]
total    = f11 + f10 + f01 + f00
accuracy = (f11 + f00) / total
error    = (f10 + f01) / total

labels     = ['f₁₁ (TP)', 'f₀₀ (TN)', 'f₁₀ (FN)', 'f₀₁ (FP)']
values     = [f11, f00, f10, f01]
bar_colors = ['#2ecc71', '#2ecc71', '#e74c3c', '#e74c3c']
bars = axes[1].bar(labels, values, color=bar_colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                str(val), ha='center', va='bottom', fontweight='bold', fontsize=13)
axes[1].set_title('Confusion Matrix Values', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Count')
axes[1].set_ylim(0, max(values) + 0.8)

plt.tight_layout()
plt.show()

print(f'\n📊 Performance Metrics')
print(f'{"="*40}')
print(f'  f₁₁ (True Positive  - TP) : {f11}')
print(f'  f₀₀ (True Negative  - TN) : {f00}')
print(f'  f₁₀ (False Negative - FN) : {f10}')
print(f'  f₀₁ (False Positive - FP) : {f01}')
print(f'  Total instances            : {total}')
print()
print(f'  ✅ Accuracy   = ({f11}+{f00})/{total} = {accuracy:.3f} = {accuracy*100:.1f}%')
print(f'  ❌ Error Rate = ({f10}+{f01})/{total} = {error:.3f} = {error*100:.1f}%')

---
<a id='4'></a>
## 4. Decision Tree Structure and Classification

### Decision Tree Components
- **Root Node:** The starting node with no incoming edges
- **Internal Nodes:** Nodes that contain attribute test conditions
- **Leaf / Terminal Nodes:** End nodes associated with a class label

In [ ]:
# Credit borrower dataset (Hunt's Algorithm example from the slides)
credit_data = {
    'ID':             range(1, 11),
    'Home_Owner':     ['Yes','No','No','Yes','No','No','Yes','No','No','No'],
    'Marital_Status': ['Single','Married','Single','Married','Divorced',
                       'Married','Divorced','Single','Married','Single'],
    'Annual_Income':  [125000, 100000, 70000, 120000, 95000,
                       60000, 220000, 85000, 75000, 90000],
    'Default':        ['No','No','No','No','Yes','No','No','Yes','No','Yes']
}
df_credit = pd.DataFrame(credit_data)
print('💰 Credit Borrower Dataset (Hunt\'s Algorithm Example)')
print('=' * 65)
print(df_credit.to_string(index=False))
print(f'\n📊 Default Distribution:')
print(df_credit['Default'].value_counts().to_string())

In [ ]:
le2 = LabelEncoder()
df_credit_enc = df_credit.copy()
df_credit_enc['Home_Owner_enc']     = le2.fit_transform(df_credit['Home_Owner'])
df_credit_enc['Marital_Status_enc'] = le2.fit_transform(df_credit['Marital_Status'])
df_credit_enc['Default_enc']        = le2.fit_transform(df_credit['Default'])

X_credit = df_credit_enc[['Home_Owner_enc', 'Marital_Status_enc', 'Annual_Income']]
y_credit = df_credit_enc['Default_enc']

clf_credit = DecisionTreeClassifier(max_depth=4, criterion='entropy', random_state=42)
clf_credit.fit(X_credit, y_credit)

fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(clf_credit,
          feature_names=['Home Owner', 'Marital Status', 'Annual Income'],
          class_names=['Default=No', 'Default=Yes'],
          filled=True, rounded=True, fontsize=10, ax=ax,
          impurity=True, precision=3)
ax.set_title('💰 Credit Borrower Decision Tree\n(Entropy Criterion, Max Depth=4)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n🌳 Tree Structure (Text Format):')
print('=' * 50)
print(export_text(clf_credit,
                  feature_names=['Home_Owner', 'Marital_Status', 'Annual_Income']))

---
<a id='5'></a>
## 5. Hunt's Algorithm Simulation

Hunt's algorithm grows the decision tree **recursively**:
1. If all records belong to the same class → create a leaf node
2. Otherwise → expand using the best splitting criterion
3. Apply recursively to each child node

In [ ]:
def compute_entropy(y):
    """Computes the entropy of a node."""
    if len(y) == 0:
        return 0
    classes, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    return -np.sum(probs * np.log2(probs + 1e-10))


def hunt_step(data, target_col, level=0, max_level=3):
    """Recursive simulation of Hunt's algorithm."""
    indent  = '  ' * level
    classes = data[target_col].unique()
    entropy = compute_entropy(data[target_col])

    print(f'{indent}📦 Node | Instances: {len(data)} | '
          f'Entropy: {entropy:.3f} | Distribution: {dict(data[target_col].value_counts())}')

    # Stopping criterion 1: all records in the same class
    if len(classes) == 1:
        print(f'{indent}  ✅ LEAF → Class: {classes[0]}')
        return

    # Stopping criterion 2: max depth reached
    if level >= max_level:
        majority = data[target_col].value_counts().idxmax()
        print(f'{indent}  🛑 Max depth → Majority class: {majority}')
        return

    # Find the best splitting attribute
    best_gain = -1
    best_attr = None

    for col in ['Home_Owner', 'Marital_Status']:
        gain = entropy
        for val in data[col].unique():
            subset = data[data[col] == val]
            weight  = len(subset) / len(data)
            gain   -= weight * compute_entropy(subset[target_col])
        if gain > best_gain:
            best_gain = gain
            best_attr = col

    # Check income threshold
    threshold = 78000
    gain_income = entropy
    for mask in [data['Annual_Income'] < threshold, data['Annual_Income'] >= threshold]:
        subset = data[mask]
        if len(subset) > 0:
            weight       = len(subset) / len(data)
            gain_income -= weight * compute_entropy(subset[target_col])
    if gain_income > best_gain:
        best_gain = gain_income
        best_attr = f'Annual_Income<{threshold}'

    print(f'{indent}  🔀 Best split: [{best_attr}] | Information Gain: {best_gain:.3f}')

    # Recursive expansion
    if best_attr and '<' not in best_attr:
        for val in sorted(data[best_attr].unique()):
            subset = data[data[best_attr] == val]
            print(f'{indent}  ↳ {best_attr} = {val}')
            hunt_step(subset, target_col, level + 1, max_level)
    else:
        left  = data[data['Annual_Income'] < threshold]
        right = data[data['Annual_Income'] >= threshold]
        print(f'{indent}  ↳ Annual_Income < {threshold}')
        hunt_step(left,  target_col, level + 1, max_level)
        print(f'{indent}  ↳ Annual_Income >= {threshold}')
        hunt_step(right, target_col, level + 1, max_level)


print('🌳 Hunt\'s Algorithm Simulation — Credit Borrower Dataset')
print('=' * 65)
hunt_step(df_credit, 'Default', max_level=2)

---
<a id='6'></a>
## 6. Attribute Types and Splitting Conditions

| Type | Description | Splitting Method |
|------|-------------|------------------|
| **Nominal** | Unordered categorical (e.g. colour, marital status) | Multi-way or binary |
| **Ordinal** | Ordered categorical (e.g. size: small/medium/large) | Preserving order |
| **Continuous** | Numeric (e.g. income, age) | Binary threshold / range query |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Nominal: Marital Status — multi-way split
ax = axes[0]
categories    = ['Single', 'Married', 'Divorced']
default_rates = [0.40, 0.00, 0.50]
bars = ax.bar(categories, default_rates,
               color=['#3498db', '#2ecc71', '#e74c3c'],
               edgecolor='white', linewidth=2)
for bar, val in zip(bars, default_rates):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{val*100:.0f}%', ha='center', fontweight='bold')
ax.set_title('📊 Nominal: Marital Status\n(Multi-way Split)', fontweight='bold')
ax.set_ylabel('Default Rate')
ax.set_ylim(0, 0.7)
ax.axhline(y=0.3, color='gray', linestyle='--', alpha=0.5, label='Overall rate')
ax.legend()

# 2. Ordinal: Shirt Size — binary split preserving order
ax = axes[1]
sizes        = ['S', 'M', 'L', 'XL']
size_colors  = ['#3498db', '#3498db', '#e74c3c', '#e74c3c']
ax.bar(sizes, [3, 4, 5, 6], color=size_colors, edgecolor='white', linewidth=2)
ax.axvline(x=1.5, color='black', linestyle='--', linewidth=2.5, label='Split point')
ax.text(0.5, 6.2, '{S,M}',   ha='center', color='#3498db', fontweight='bold', fontsize=12)
ax.text(2.5, 6.2, '{L,XL}',  ha='center', color='#e74c3c', fontweight='bold', fontsize=12)
ax.set_title('📏 Ordinal: Shirt Size\n(Binary Split Preserving Order)', fontweight='bold')
ax.set_ylabel('Count')
ax.legend()
ax.set_ylim(0, 7.5)

# 3. Continuous: Annual Income
ax = axes[2]
np.random.seed(42)
income_no  = np.random.normal(55000, 20000, 30)
income_yes = np.random.normal(95000, 25000, 15)
ax.hist(income_no,  bins=10, alpha=0.6, color='#3498db', label='Default=No',  edgecolor='white')
ax.hist(income_yes, bins=10, alpha=0.6, color='#e74c3c', label='Default=Yes', edgecolor='white')
ax.axvline(x=78000, color='black', linestyle='--', linewidth=2.5, label='Threshold: $78K')
ax.set_title('💵 Continuous: Annual Income\n(Binary Split with Threshold)', fontweight='bold')
ax.set_xlabel('Annual Income ($)')
ax.set_ylabel('Frequency')
ax.legend(fontsize=9)

plt.suptitle('Attribute Types and Splitting Conditions', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n💡 Summary:')
print('  • Nominal attributes  → Multi-way or binary split')
print('  • Ordinal attributes  → Grouping while preserving order')
print('  • Continuous attributes → A < v (binary) or vi ≤ A < vi+1 (multi-way range)')

---
<a id='7'></a>
## 7. Impurity Measures: Entropy, Gini Index, Classification Error

$$\\text{Entropy} = -\\sum_{i=0}^{c-1} p_i(t) \\log_2 p_i(t)$$

$$\\text{Gini} = 1 - \\sum_{i=0}^{c-1} p_i(t)^2$$

$$\\text{Classification Error} = 1 - \\max_i[p_i(t)]$$

In [ ]:
def entropy(p):
    """Entropy for binary classification. p = P(Class=1)"""
    if p == 0 or p == 1:
        return 0.0
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)

def gini(p):
    """Gini index for binary classification."""
    return 1 - p**2 - (1 - p)**2

def classification_error(p):
    """Classification error."""
    return 1 - max(p, 1 - p)

p_values    = np.linspace(0, 1, 500)
entropy_vals = [entropy(p)              for p in p_values]
gini_vals    = [gini(p)                 for p in p_values]
error_vals   = [classification_error(p) for p in p_values]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: comparison of all three measures
axes[0].plot(p_values, entropy_vals, 'b-',  linewidth=2.5, label='Entropy')
axes[0].plot(p_values, gini_vals,    'r--', linewidth=2.5, label='Gini Index')
axes[0].plot(p_values, error_vals,   'g-.', linewidth=2.5, label='Classification Error')
axes[0].axvline(x=0.5, color='gray', linestyle=':', alpha=0.7, label='p=0.5 (max impurity)')
axes[0].set_xlabel('p (proportion of Class 1)', fontsize=12)
axes[0].set_ylabel('Impurity Value', fontsize=12)
axes[0].set_title('Impurity Measures Comparison', fontweight='bold', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Right: slide node examples N1, N2, N3
nodes      = ['N1\n(0/6, 6/6)', 'N2\n(1/6, 5/6)', 'N3\n(3/6, 3/6)']
p_examples = [1.0, 5/6, 0.5]
e_vals = [entropy(p)              for p in p_examples]
g_vals = [gini(p)                 for p in p_examples]
h_vals = [classification_error(p) for p in p_examples]

x = np.arange(len(nodes))
w = 0.25
axes[1].bar(x - w, e_vals, w, label='Entropy', color='#3498db', edgecolor='white')
axes[1].bar(x,     g_vals, w, label='Gini',    color='#e74c3c', edgecolor='white')
axes[1].bar(x + w, h_vals, w, label='Error',   color='#2ecc71', edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(nodes, fontsize=10)
axes[1].set_ylabel('Impurity Value')
axes[1].set_title('Node Examples (N1, N2, N3)', fontweight='bold', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.3f', fontsize=8, padding=2)

plt.suptitle('Impurity Measures: Entropy, Gini Index, Classification Error',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print('📐 Impurity Measure Calculations (Slide Examples)')
print('=' * 60)

nodes_data = [
    ('N1', 0, 6),   # Class=0: 0 instances, Class=1: 6 instances
    ('N2', 1, 5),   # Class=0: 1 instance,  Class=1: 5 instances
    ('N3', 3, 3),   # Class=0: 3 instances, Class=1: 3 instances
]

for name, c0, c1 in nodes_data:
    total   = c0 + c1
    p0, p1  = c0 / total, c1 / total
    p       = p1

    gini_val  = 1 - p0**2 - p1**2
    ent_val   = entropy(p)
    err_val   = classification_error(p)

    print(f'\n{"─"*50}')
    print(f'  Node {name}: Class=0: {c0} instances, Class=1: {c1} instances')
    print(f'  p₀={p0:.4f}, p₁={p1:.4f}')
    print(f'  Gini  = 1 - ({p0:.3f})² - ({p1:.3f})² = {gini_val:.3f}')
    print(f'  Entropy              = {ent_val:.3f}')
    print(f'  Error = 1 - max[{p0:.3f}, {p1:.3f}] = {err_val:.3f}')

print(f'\n{"─"*50}')
print('\n💡 Observation: N1 is perfectly pure (all instances same class) → all measures = 0')
print('               N3 is maximally impure (equal split) → all measures at maximum')

---
<a id='8'></a>
## 8. Information Gain and Gain Ratio

$$\\Delta_{info} = I(\\text{parent}) - I(\\text{children})$$

$$\\text{Gain Ratio} = \\frac{\\Delta_{info}}{\\text{Split Information}}$$

$$\\text{Split Information} = -\\sum_{j=1}^{k} \\frac{N(v_j)}{N} \\log_2 \\frac{N(v_j)}{N}$$

In [ ]:
def information_gain(data, attribute, target):
    """Computes the information gain for a given attribute."""
    parent_entropy = compute_entropy(data[target])
    weighted_entropy = sum(
        len(data[data[attribute] == v]) / len(data) *
        compute_entropy(data[data[attribute] == v][target])
        for v in data[attribute].unique()
    )
    return parent_entropy - weighted_entropy, weighted_entropy

def split_information(data, attribute):
    """Computes the split information for a given attribute."""
    si = 0
    for val in data[attribute].unique():
        ratio = len(data[data[attribute] == val]) / len(data)
        if ratio > 0:
            si -= ratio * np.log2(ratio)
    return si

parent_entropy = compute_entropy(df_credit['Default'])

print('📊 Information Gain Calculation — Credit Borrower Dataset')
print('=' * 65)
print(f'  Parent node entropy I(parent) = {parent_entropy:.3f}')
print(f'  Distribution: {dict(df_credit["Default"].value_counts())}\n')

for attr in ['Home_Owner', 'Marital_Status']:
    gain, weighted = information_gain(df_credit, attr, 'Default')
    si             = split_information(df_credit, attr)
    gain_ratio     = gain / si if si > 0 else 0
    print(f'  {attr}:')
    print(f'    Weighted child entropy I(children) = {weighted:.3f}')
    print(f'    Information Gain Δ                 = {parent_entropy:.3f} - {weighted:.3f} = {gain:.3f}')
    print(f'    Split Information                  = {si:.3f}')
    print(f'    Gain Ratio                         = {gain:.3f} / {si:.3f} = {gain_ratio:.3f}')
    print()

In [ ]:
# FIX: Gender column shuffled so it does NOT perfectly separate classes.
# The original ['M']*10 + ['F']*10 gave Gender an info gain of 1.0 (same as CustomerID),
# making the gain ratio comparison pointless. The corrected version gives Gender gain ≈ 0.
customer_data = {
    'CustomerID': range(1, 21),
    'Gender':  ['M','M','M','F','F','M','F','M','F','F',
                'F','F','F','M','M','F','M','M','M','F'],
    'CarType': ['Family', 'Sports', 'Sports', 'Sports', 'Sports', 'Sports',
                'Sports', 'Sports', 'Sports', 'Luxury',
                'Family', 'Family', 'Family', 'Luxury', 'Luxury',
                'Luxury', 'Luxury', 'Luxury', 'Luxury', 'Luxury'],
    'Class':   ['C0']*10 + ['C1']*10
}
df_customer = pd.DataFrame(customer_data)

parent_e = compute_entropy(df_customer['Class'])

print('🚗 Gain Ratio Example: Gender, Car Type, Customer ID')
print('=' * 65)
print(f'  Parent node entropy = {parent_e:.3f} (10 C0, 10 C1 → equal split)\n')

for attr, label in [('Gender', 'Gender'), ('CarType', 'Car Type')]:
    gain, weighted = information_gain(df_customer, attr, 'Class')
    si             = split_information(df_customer, attr)
    gr             = gain / si if si > 0 else 0
    print(f'  {label} ({attr}):')
    print(f'    Child entropy          = {weighted:.3f}')
    print(f'    Information Gain       = {parent_e:.3f} - {weighted:.3f} = {gain:.3f}')
    print(f'    Split Information      = {si:.3f}')
    print(f'    Gain Ratio             = {gain:.3f} / {si:.3f} = {gr:.3f}')
    print()

gain_id    = parent_e - 0      # Perfect split: each leaf has one instance
si_id      = np.log2(20)       # -20 × (1/20) × log2(1/20) = log2(20)
gr_id      = gain_id / si_id
print(f'  Customer ID:')
print(f'    Child entropy          = 0.000 (one instance per leaf)')
print(f'    Information Gain       = {gain_id:.3f} (maximum!)')
print(f'    Split Information      = {si_id:.3f}')
print(f'    Gain Ratio             = {gain_id:.3f} / {si_id:.3f} = {gr_id:.3f}')
print()
print('💡 Key insight: CustomerID has the highest information gain but the lowest gain ratio')
print('   because it creates too many splits → risk of overfitting!')

In [ ]:
attributes   = ['Gender', 'CarType\n(Car Type)', 'CustomerID']
info_gains   = []
gain_ratios  = []

for attr in ['Gender', 'CarType']:
    g, _  = information_gain(df_customer, attr, 'Class')
    si    = split_information(df_customer, attr)
    info_gains.append(g)
    gain_ratios.append(g / si if si > 0 else 0)

info_gains.append(gain_id)
gain_ratios.append(gr_id)

x = np.arange(len(attributes))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - w/2, info_gains,  w, label='Information Gain (ΔInfo)',
               color='#3498db', edgecolor='white', linewidth=1.5)
bars2 = ax.bar(x + w/2, gain_ratios, w, label='Gain Ratio',
               color='#e74c3c', edgecolor='white', linewidth=1.5)

ax.bar_label(bars1, fmt='%.3f', fontsize=10, padding=3, fontweight='bold')
ax.bar_label(bars2, fmt='%.3f', fontsize=10, padding=3, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(attributes, fontsize=11)
ax.set_ylabel('Value', fontsize=12)
ax.set_title('Information Gain vs Gain Ratio\nWhy should we NOT split on CustomerID?',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1.1)

ax.annotate('CustomerID: High gain\nbut low gain ratio!\n→ Overfitting risk',
            xy=(2, gr_id), xytext=(1.35, 0.5),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=9, bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

plt.tight_layout()
plt.show()

---
<a id='9'></a>
## 9. Overfitting and Underfitting

- **Underfitting:** Model too simple → both training and test error are high
- **Overfitting:** Model memorises training data → training error ≈ 0 but test error rises
- **Ideal Model:** Balanced tree depth → low training and test error

In [ ]:
from sklearn.datasets import make_classification

np.random.seed(42)
X_ovf, y_ovf = make_classification(n_samples=300, n_features=10,
                                     n_informative=5, n_redundant=2,
                                     random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_ovf, y_ovf, test_size=0.33, random_state=42)

depths       = range(1, 20)
train_errors = []
test_errors  = []

for d in depths:
    clf_d = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf_d.fit(X_tr, y_tr)
    train_errors.append(1 - clf_d.score(X_tr, y_tr))
    test_errors.append(1  - clf_d.score(X_te, y_te))

best_depth = list(depths)[np.argmin(test_errors)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(depths), train_errors, 'b-o', linewidth=2,
             markersize=5, label='Training Error')
axes[0].plot(list(depths), test_errors,  'r-s', linewidth=2,
             markersize=5, label='Test Error')
axes[0].axvline(x=best_depth, color='green', linestyle='--',
                linewidth=2, label=f'Best Depth={best_depth}')
axes[0].fill_betweenx([0, max(test_errors)+0.05],
                       1, 2.5, alpha=0.1, color='orange', label='Underfitting')
axes[0].fill_betweenx([0, max(test_errors)+0.05],
                       12, 20, alpha=0.1, color='red',    label='Overfitting')
axes[0].set_xlabel('Tree Depth', fontsize=11)
axes[0].set_ylabel('Error Rate', fontsize=11)
axes[0].set_title('Tree Depth vs Error Rate', fontweight='bold', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].axis('off')
table_data = [
    ['Underfitting', 'Too Small (1-2)',          'High',  'High'],
    ['Ideal Model',  f'Balanced (~{best_depth})', 'Low',   'Low'],
    ['Overfitting',  'Too Large (15+)',           '≈ 0',   'Rising'],
]
col_labels = ['Condition', 'Tree Size', 'Training Error', 'Test Error']
table = axes[1].table(cellText=table_data, colLabels=col_labels,
                       loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2.2)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    elif row == 1:
        cell.set_facecolor('#fdebd0')
    elif row == 2:
        cell.set_facecolor('#d5f5e3')
    else:
        cell.set_facecolor('#fce4e4')
axes[1].set_title('Underfitting — Ideal — Overfitting', fontweight='bold', fontsize=12)

plt.suptitle('Model Complexity and Generalisation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n📊 Optimal depth  : {best_depth}')
print(f'   Training error : {train_errors[best_depth-1]:.3f}')
print(f'   Test error     : {test_errors[best_depth-1]:.3f}')

In [ ]:
print('📐 Complexity Estimate: errgen(T) = err(T) + Ω × (k / N_train)')
print('=' * 65)
print('Slide example: TL (7 leaves) vs TR (4 leaves), N=24 training instances')
print()

N = 24
kL, kR   = 7, 4
errL, errR = 4/N, 6/N

for omega in [0.5, 1.0, 0.25]:
    errgen_L = errL + omega * (kL / N)
    errgen_R = errR + omega * (kR / N)
    preferred = 'TL' if errgen_L < errgen_R else 'TR'
    print(f'  Ω = {omega}:')
    print(f'    errgen(TL) = {errL:.3f} + {omega} × ({kL}/{N}) = {errgen_L:.4f}')
    print(f'    errgen(TR) = {errR:.3f} + {omega} × ({kR}/{N}) = {errgen_R:.4f}')
    print(f'    → Preferred model: {preferred}')
    print()

omega_vals    = np.linspace(0, 2, 200)
errgen_L_vals = errL + omega_vals * (kL / N)
errgen_R_vals = errR + omega_vals * (kR / N)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(omega_vals, errgen_L_vals, 'b-', linewidth=2.5, label=f'TL (k={kL} leaves)')
ax.plot(omega_vals, errgen_R_vals, 'r-', linewidth=2.5, label=f'TR (k={kR} leaves)')

omega_cross = (errR - errL) / ((kL - kR) / N)
ax.axvline(x=omega_cross, color='green', linestyle='--', linewidth=2,
           label=f'Crossover Ω = {omega_cross:.2f}')
ax.scatter([omega_cross], [errL + omega_cross * (kL/N)], s=150, zorder=5, color='green')

y_min = min(errgen_L_vals.min(), errgen_R_vals.min())
y_max = max(errgen_L_vals.max(), errgen_R_vals.max())
ax.fill_betweenx([y_min, y_max], 0, omega_cross,
                  alpha=0.1, color='blue', label='TL preferred')
ax.fill_betweenx([y_min, y_max], omega_cross, 2,
                  alpha=0.1, color='red',  label='TR preferred')

ax.set_xlabel('Ω (Hyperparameter)', fontsize=12)
ax.set_ylabel('Estimated Generalisation Error', fontsize=12)
ax.set_title('Model Preference as a Function of Ω', fontweight='bold', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 2)
plt.tight_layout()
plt.show()

---
<a id='10'></a>
## 10. Model Selection and Cross-Validation

**Holdout Method:** Split data into training and test sets.

**Cross-Validation:** Partition data into k folds; use a different fold as the test set each time.

In [ ]:
from sklearn.datasets import make_classification

np.random.seed(42)
X_all, y_all = make_classification(n_samples=500, n_features=10,
                                    n_informative=5, random_state=42)

X_Dtrain, X_Dtest, y_Dtrain, y_Dtest = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42)

X_Dtr, X_Dval, y_Dtr, y_Dval = train_test_split(
    X_Dtrain, y_Dtrain, test_size=0.33, random_state=42)

print('📊 Data Split Summary')
print('=' * 45)
print(f'  Full dataset (D)         : {len(X_all)} instances')
print(f'  D.train                  : {len(X_Dtrain)} instances')
print(f'    D.tr   (training)      : {len(X_Dtr)} instances')
print(f'    D.val  (validation)    : {len(X_Dval)} instances')
print(f'  D.test   (final test)    : {len(X_Dtest)} instances')
print()

depth_range        = range(1, 15)
validation_errors  = []

for d in depth_range:
    clf_d = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf_d.fit(X_Dtr, y_Dtr)
    validation_errors.append(1 - clf_d.score(X_Dval, y_Dval))

optimal_p = list(depth_range)[np.argmin(validation_errors)]
print(f'✅ Optimal hyperparameter p* = max_depth={optimal_p}')
print(f'   Validation error errval(p*) = {min(validation_errors):.3f}')

clf_final = DecisionTreeClassifier(max_depth=optimal_p, random_state=42)
clf_final.fit(X_Dtrain, y_Dtrain)   # Retrain on all of D.train
test_error = 1 - clf_final.score(X_Dtest, y_Dtest)
print(f'   Final test error errtest     = {test_error:.3f}')

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(list(depth_range), validation_errors, 'r-o',
        linewidth=2, markersize=6, label='Validation Error errval(p)')
ax.axvline(x=optimal_p, color='green', linestyle='--', linewidth=2.5,
           label=f'p* = {optimal_p} (optimal)')
ax.scatter([optimal_p], [min(validation_errors)], s=200, zorder=5,
           color='green', label=f'Min errval = {min(validation_errors):.3f}')
ax.set_xlabel('Hyperparameter p (max_depth)', fontsize=12)
ax.set_ylabel('Validation Error Rate', fontsize=12)
ax.set_title('Hyperparameter Selection: Optimal Depth via Validation Set',
             fontweight='bold', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# FIX: replaced the escaped-quote f-string (SyntaxError in Python 3.10)
# with a clean intermediate variable.
print('🔄 Cross-Validation — 3-Fold Example')
print('=' * 60)
print('Data is partitioned into 3 equal folds (S1, S2, S3):\n')

kf          = KFold(n_splits=3, shuffle=True, random_state=42)
fold_errors = []
all_folds   = {1, 2, 3}

for fold, (train_idx, test_idx) in enumerate(kf.split(X_Dtrain), 1):
    X_fold_tr = X_Dtrain[train_idx]
    X_fold_te = X_Dtrain[test_idx]
    y_fold_tr = y_Dtrain[train_idx]
    y_fold_te = y_Dtrain[test_idx]

    clf_fold = DecisionTreeClassifier(max_depth=optimal_p, random_state=42)
    clf_fold.fit(X_fold_tr, y_fold_tr)
    error = 1 - clf_fold.score(X_fold_te, y_fold_te)
    fold_errors.append(error)

    other_folds  = sorted(all_folds - {fold})
    train_str    = 'S' + '+S'.join(str(f) for f in other_folds)
    print(f'  Run {fold}: Train={train_str}, Test=S{fold} → Error: {error:.3f}')

mean_error = np.mean(fold_errors)
std_error  = np.std(fold_errors)
print(f'\n  📊 Mean CV Error: {mean_error:.3f} ± {std_error:.3f}')

cv_scores = cross_val_score(
    DecisionTreeClassifier(max_depth=optimal_p, random_state=42),
    X_Dtrain, y_Dtrain, cv=5
)
print(f'\n  🔁 5-Fold CV Accuracies: {cv_scores.round(3)}')
print(f'  📊 Mean: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

---
<a id='11'></a>
## 11. Model Comparison — Statistical Test

When comparing two models:
$$\\sigma^2_d \\approx \\frac{e_A(1-e_A)}{n_A} + \\frac{e_B(1-e_B)}{n_B}$$

$$d_t = d \\pm z_{\\alpha/2} \\hat{\\sigma}_d$$

If the confidence interval **contains 0** → the difference is not statistically significant.

In [ ]:
from scipy import stats

def compare_models(eA, nA, eB, nB, confidence=0.95):
    """
    Statistically compares two models.
    eA, eB : error rates  |  nA, nB : test set sizes
    """
    d        = eA - eB
    sigma2_d = (eA * (1 - eA) / nA) + (eB * (1 - eB) / nB)
    sigma_d  = np.sqrt(sigma2_d)
    alpha    = 1 - confidence
    z        = stats.norm.ppf(1 - alpha / 2)
    lower    = d - z * sigma_d
    upper    = d + z * sigma_d
    significant = not (lower <= 0 <= upper)
    return {'d': d, 'sigma_d': sigma_d, 'z': z,
            'lower': lower, 'upper': upper, 'significant': significant}

# MA: 85% accuracy → error rate eA=0.15 | MB: 75% accuracy → error rate eB=0.25
print('📊 Slide Model Comparison Example')
print('=' * 55)
print('  MA: 85% accuracy, 30   test instances → eA = 0.15')
print('  MB: 75% accuracy, 5000 test instances → eB = 0.25')
print()

result = compare_models(eA=0.15, nA=30, eB=0.25, nB=5000)

# FIX: d is negative (MA has lower error); show both signed d and |d|
print(f'  d = eA - eB = 0.15 - 0.25 = {result["d"]:.3f}')
print(f'  |d| = {abs(result["d"]):.3f}  (MA is better by this margin)')
print(f'  σ²_d = 0.15×0.85/30 + 0.25×0.75/5000 = {result["sigma_d"]**2:.4f}')
print(f'  σ_d  = {result["sigma_d"]:.4f}')
print(f'  z_{{α/2}} = {result["z"]:.3f}  (95% confidence)')
print(f'  dt = {result["d"]:.3f} ± {result["z"]:.3f} × {result["sigma_d"]:.4f}')
print(f'     = {result["d"]:.3f} ± {result["z"]*result["sigma_d"]:.4f}')
print(f'  Confidence Interval: [{result["lower"]:.3f}, {result["upper"]:.3f}]')
print()
if result['significant']:
    print('  ✅ Conclusion: The difference IS statistically significant.')
else:
    print('  ❌ Conclusion: The difference is NOT statistically significant.')
    print('  The CI contains 0 → we cannot claim MA is truly better.')
    print(f'  Why? MA was tested on only {30} instances → high uncertainty (wide CI).')

In [ ]:
print('\n📐 Effect of Sample Size on the Confidence Interval')
print('=' * 55)
print('MA accuracy fixed at 85%; test set size varies:\n')

sizes = [30, 100, 500, 1000, 5000]
for n in sizes:
    r     = compare_models(eA=0.15, nA=n, eB=0.25, nB=5000)
    width = r['upper'] - r['lower']
    label = '✅ Significant' if r['significant'] else '❌ Not significant'
    print(f'  n={n:5d}: CI=[{r["lower"]:+.3f}, {r["upper"]:+.3f}]  '
          f'Width={width:.3f}  {label}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, n in enumerate(sizes):
    r     = compare_models(eA=0.15, nA=n, eB=0.25, nB=5000)
    color = '#2ecc71' if r['significant'] else '#e74c3c'
    axes[0].errorbar(i, r['d'], yerr=r['z']*r['sigma_d'],
                     fmt='o', color=color, capsize=8, capthick=2,
                     markersize=8, linewidth=2)
    axes[0].text(i, r['d'] + r['z']*r['sigma_d'] + 0.01,
                '✅' if r['significant'] else '❌', ha='center', fontsize=14)

axes[0].axhline(y=0, color='black', linestyle='--', linewidth=2, label='d=0 (no difference)')
axes[0].set_xticks(range(len(sizes)))
axes[0].set_xticklabels([f'n={n}' for n in sizes], fontsize=10)
axes[0].set_ylabel('Error Difference (d = eA − eB)', fontsize=11)
axes[0].set_title('Sample Size and Confidence Interval\n(95% Confidence Level)',
                   fontweight='bold', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

widths  = []
n_range = range(10, 2000, 20)
for n in n_range:
    r = compare_models(eA=0.15, nA=n, eB=0.25, nB=5000)
    widths.append(r['upper'] - r['lower'])

axes[1].plot(list(n_range), widths, 'b-', linewidth=2)
axes[1].set_xlabel('MA Test Set Size (nA)', fontsize=11)
axes[1].set_ylabel('Confidence Interval Width', fontsize=11)
axes[1].set_title('Larger Sample → Narrower CI → Less Uncertainty',
                   fontweight='bold', fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].fill_between(list(n_range), widths, alpha=0.2, color='blue')

plt.suptitle('Statistical Model Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📚 Summary — Key Concepts

| Concept | Description |
|---------|-------------|
| **Classification** | Learning a model from (x, y) pairs to predict y for new x |
| **Induction** | Building a model from training data |
| **Deduction** | Applying the learned model to test data |
| **Confusion Matrix** | Performance summary built from f₁₁, f₁₀, f₀₁, f₀₀ |
| **Entropy** | Measures uncertainty in a node; 0 = pure, max = equal split |
| **Gini Index** | Probability of misclassification; 0 = pure |
| **Information Gain** | Parent entropy − weighted child entropy |
| **Gain Ratio** | Normalises information gain by the number of splits |
| **Overfitting** | Model memorisation: training error↓, test error↑ |
| **Underfitting** | Model too simple: both errors↑ |
| **Cross-Validation** | Reliable method to estimate generalisation performance |
| **Confidence Interval** | Statistical significance test for comparing two models |

---
> 💡 **Next Lecture:** Other classification algorithms (k-NN, Naive Bayes, SVM, etc.)